In [ ]:
import os
import shutil
import sys

import pandas as pd
from pynxtools_em import get_pynxtools_em_version
from pynxtools_em.examples.get_file_from_archive_formats import (
    get_file_from_rar,
    get_file_from_tar,
    get_file_from_zip,
    get_file_from_sevenzip
)
from pynxtools_em.examples.get_sha256_of_directories import SEPARATOR
from pynxtools_em.examples.oasisb_utils import get_project_id, prepare_em_ebsd_mtex  # EM_EBSD_MTEX_MIME_TYPES_SIDECAR, EM_EBSD_MTEX_MIME_TYPES_SOLITARY, CSV_HEADER_FOR_HASH_FILE

print(os.getcwd())
with open("source_directory.txt") as fp:
    src_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(src_directory)
with open("target_directory.txt") as fp:
    trg_directory = f"{fp.readline().strip().replace('/', os.sep)}"
print(trg_directory)

## Decompress the original files from the scientists from the storage location

Locally, original research data are stored compressed when not needed.<br>
Maybe multiple compressed files per project directory.<br>

In [ ]:
config: dict[str, str] = {
    "python_version": f"{sys.version.replace(' ', '_')}",
    "working_directory": f"{os.getcwd()}",
    "pynxtools_em version": f"{get_pynxtools_em_version()}",
    # "directory": f"src_directory,  # sys.argv[1],
}

spread_sheet_of_all_projects = pd.read_excel(
    f"{src_directory}{os.sep}aaa_legacy_data.ods",
    sheet_name="aaa_legacy_data",
    engine="odf",
    dtype=str,
).fillna("")
project_range: tuple[int, int] = (1, 836)

# keep_searching_toggle = True
cnt = 0
for row in spread_sheet_of_all_projects.itertuples(index=True):
    if row.project_name != "" and row.use == "1" and row.legal == "1":
        # row.parse == 2 if really only cross-ref data if available if dataset is CC0-1.0 or CC-BY-4.0
        # row.parse in (1, 2) if also allowing locally shared datasets, these will not be uploaded to any public deployment though
        if project_range[0] <= int(row.project_name) <= project_range[1]:
            project_id = get_project_id(f"{row.project_name}")
            print(f"project{SEPARATOR}{project_id}{SEPARATOR}decompress...")

            status = prepare_em_ebsd_mtex(
                f"{src_directory}{os.sep}{project_id}.sha256.results.csv",
                project_id,
                trg_directory
            )
            for key, obj in status.items():
                if obj["n"] > 0:
                    print(f"{key}, {obj['n']}")
                    cnt += obj['n']
print(cnt)